In [1]:
%pip install jupysql duckdb duckdb-engine --quiet

%load_ext sql
%sql duckdb:///:memory:

Note: you may need to restart the kernel to use updated packages.


Connecting to 'duckdb:///:memory:'

In [2]:
%%sql
create table bookings as select * from read_csv_auto('bookings.csv');
create table clients as select * from read_csv_auto('clients.csv');

Running query in 'duckdb:///:memory:'

Count
50


In [30]:
%%sql
with h1_clients as (
    select 
        distinct client_id
    from bookings
    where booking_date >= date '2025-01-01' and booking_date < date '2025-07-01' and status = 'confirmed'
),
h2_clients as (
    select 
        distinct client_id
    from bookings
    where booking_date >= date '2025-07-01' and booking_date < date '2026-01-01' and status = 'confirmed'
),
churn_clients as (
    select 
        client_id 
    from h1_clients
    where client_id not in (select * from h2_clients)
),
h1_gmv as (
    select
        client_id,
        round(sum(amount_rub),2) as h1_gmv
    from bookings
    where booking_date >= date '2025-01-01' and booking_date < date '2025-07-01' and status = 'confirmed'
    group by client_id
),

churn_clients_h1_gmv as (
    select 
        a.client_id as client_id, 
        h1_gmv
    from churn_clients as a left join h1_gmv as b
    on a.client_id = b.client_id
),

churn_clients_h1_gmv_tier as (
    select 
        a.client_id as client_id, 
        h1_gmv,
        tier
    from churn_clients_h1_gmv as a left join clients as b
        on a.client_id = b.client_id
        order by h1_gmv desc
)

select 
    tier,
    count(*) as clients_count, 
    round(sum(h1_gmv),2) as H1_GMV
from churn_clients_h1_gmv_tier
group by tier
order by tier 

Running query in 'duckdb:///:memory:'

tier,clients_count,H1_GMV
2,6,13193157.78
3,10,4316954.02


In [ ]:
Всего 16 клиентов: 6 клиентов второго тира и 10 клиентов третьего тира.
Суммарный GMV за H1: 13193157.78 + 4316954.02 = 17510111.80 RUB